In [7]:
import pathlib as pl

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]
DATA_ROOT = CONFIG["project_data"]

### begin custom

import pandas as pd

compare_to = "ar"

if compare_to == "ar":
    theirs_file = CONFIG["project_repo"].joinpath(
        "annotation", "norm", "20250404_arhie_chry_seqs.v2.tsv"
    ).resolve(strict=True)
elif compare_to == "ml":
    theirs_file = CONFIG["project_repo"].joinpath(
        "annotation", "raw", "ml_genes.tsv"
    ).resolve(strict=True)
elif compare_to == "ar-ml":
    theirs_file = CONFIG["project_repo"].joinpath(
        "annotation", "raw", "20250325_T2T-Y_Scaffolds_PathAndNodes.tsv"
    ).resolve(strict=True)
else:
    raise
ours_file = CONFIG["project_repo"].joinpath(
    "codebase", "extract-sex-chromosomes", "workflow", "results",
    "seq_rename_composition.tsv"
).resolve(strict=True)

ours = pd.read_csv(ours_file, sep="\t", header=0, comment="#")
ours = ours.loc[ours["assigned_chrom"] == "chrY", :].copy()

sample_remap = {
    "HG002": "NA24385",
    "HG005": "NA24631",
    "HG003": "NA24149"
}

theirs = pd.read_csv(theirs_file, sep="\t", header=0, comment="#")
theirs["sample"] = theirs["sample"].replace(sample_remap)
if compare_to == "ar":
    print(theirs.shape)
    theirs = theirs.loc[theirs["proc_status"] != "manual", :].copy()
    print(theirs.shape)
    exclude_list = """
        HG01167
        HG01192
        HG02486
        HG02514
        HG03139
        HG03688
        HG04187
        NA18608
        NA18971
        NA18974
    """
    exclude_list = exclude_list.strip().split()
    for sample in exclude_list:
        if sample in theirs["sample"]:
            print("still in ", sample)
    ours = ours.loc[~ours["sample"].isin(exclude_list), :].copy()
    
print(theirs.shape)



matchings = []
ours_indices = []
for row in theirs.itertuples():
    if compare_to == "ar-ml":
        select_sample = ours["sample"] == row.sample
        try:
            select_long = ours["source_name"] == "chrY_" + row.Seq
        except TypeError:
            # these are the NA sequences
            categ = "no-match"
            our_idx = -1
            matchings.append((row.Index, categ, our_idx))
            continue
    else:
        select_sample = ours["sample"] == row.sample
        #select_long = ours["source_name"] == row.tig
        select_long = ours["source_name"] == row.old_name
    if compare_to == "ar":
        select_short = ours["source_name"] == "chrY_" + row.old_name
        select_any = select_sample & (select_long | select_short)
    if compare_to == "ar-ml":
        select_short = ours["source_name"] == row.Seq
        select_any = select_sample & (select_long | select_short)
    if compare_to == "ml":
        select_any = select_sample & select_long
    if not select_any.any():
        categ = "no-match"
        our_idx = -1
    elif select_any.sum() > 2:
        categ = "multi-match"
        our_idx = -2
    else:
        our_idx = ours.index[select_any].values[0]
        categ = "match"
        ours_indices.append(our_idx)
    matchings.append((row.Index, categ, our_idx))

matchings = pd.DataFrame.from_records(
    matchings, columns=["idx", "categ", "ours_idx"]
)

theirs = theirs.merge(matchings, left_index=True, right_on="idx")

categ_counts = theirs["categ"].value_counts()
print(categ_counts)

theirs_file_out = theirs_file.with_suffix(".matchings.tsv")
theirs.to_csv(theirs_file_out, sep="\t", header=True, index=False)
theirs_no_match = theirs.loc[theirs["categ"] == "no-match", :].copy()
print(theirs_no_match.shape)
print(theirs_no_match.head())

inverse = ~ours.index.isin(ours_indices)
sub = ours.loc[inverse, :].copy()
# HG02015 is misassembled
sub = sub.loc[~sub["sample"].isin(["HG02015", "HG02145"]), :].copy()
print(sub["seq_length"].describe())
print(sub.shape)

print("long and missing")
for row in sub.loc[sub["seq_length"] > int(2e5), :].itertuples():
    if row.sample in exclude_list:
        continue
    print(row.sample, row.source_name, row.seq_length)

(813, 6)
(797, 6)
(797, 6)
categ
match       691
no-match    106
Name: count, dtype: int64
(106, 9)
      sample    alias            old_name                    new_name  orient  \
175  HG01106  HG01106    chrX_mat-0000007                HG01106_chrY       1   
177  HG01106  HG01106         mat-0000007  HG01106_chrY_random0000007       0   
182  HG01109  HG01109  unassigned-0000932  HG01109_chrY_random0000932       0   
187  HG01192  HG01192  haplotype1-0000014  HG01192_chrY_random0000014       0   
188  HG01192  HG01192  haplotype1-0000015  HG01192_chrY_random0000015       0   

    proc_status  idx     categ  ours_idx  
175        auto  175  no-match        -1  
177        auto  177  no-match        -1  
182        auto  182  no-match        -1  
187        auto  189  no-match        -1  
188        auto  190  no-match        -1  
count       500.000000
mean      88183.286000
std       67961.997324
min        3320.000000
25%       38934.250000
50%       71759.500000
75%      123549.5